## Run the cell below to copy a new overlay form /home/xilinx 

In [23]:
! mv /home/xilinx/main.* .

## The code below starts a waveform

In [24]:
from pynq import Overlay, MMIO, allocate
from xrfdc import RFdc
from pynq.overlays.base import BaseOverlay
import time
import xrfclk
import numpy as np
# import os
# import glob
# import re
# import struct
# from collections import defaultdict

            



class ESBL(BaseOverlay):
    

    
    def __init__(self,NCO=False,ext_clock=True):
        Overlay.__init__(self,'main.bit')
        if self.is_loaded():
            self.i2c_initialized = False
            self.display_port_initialized = False
            self.radio = self.usp_rf_data_converter_0
            self.reg = MMIO(self.ip_dict['ESBL_control_reg']['phys_addr'],
                            self.ip_dict['ESBL_control_reg']['addr_range']).array
            
            
            self.xrfdc_alias = MMIO(self.ip_dict['usp_rf_data_converter_0']['phys_addr'],
                             self.ip_dict['usp_rf_data_converter_0']['addr_range']).array
            
            self.init_rf_clks()
            if ext_clock:
                self.init_external_rf_clks() # Must run after self.init_rf_clks()
            
            if NCO:
                self.freq = 0.001
    #             self.radio.dac_tiles[0].blocks[0].MixerSettings['EventSource'] = 0 # Set to Immediate
                self.radio.dac_tiles[0].blocks[0].MixerSettings['EventSource'] = 2 # Set to Tile (necessary for DMA)
                self.radio.dac_tiles[0].blocks[0].MixerSettings['FineMixerScale'] = 1
                self.freq = 245.0
            
            self.set_i_ampl()
            self.set_q_ampl()
            self.set_phase_difference()
            self.reg[6] = 1
            
    def init_external_rf_clks(self,lmk_freq=245.76):
        # Setting the clock input of the LMK chip to CLKin0, which is the CLK_IN pin on RFSoC 4x2
        
#         self.Config = defaultdict(dict)
#         self.ex_read_tics_output()
        
        for lmk in xrfclk.lmk_devices:
            with open(lmk['spi_device'], 'rb+', buffering=0) as f:
                # for v in self.Config[lmk['compatible']][lmk_freq]:
                #     data = struct.pack('>I', v)
                data = b'\x00\x01\x47\x0A' # Only works for LMK0482x series
                if lmk['num_bytes'] == 3:
                    f.write(data[1:])
                else:
                    f.write(data)
            
    def trigger_DMA(self):
        self.reg[6]=0
        self.reg[6]=1
    
    # def ex_read_tics_output(self):
    #     ex_lmk_txt = glob.glob('*.txt')
    #     print(ex_lmk_txt)
    #     for s in ex_lmk_txt:
    #         chip, freq = s.lower().split('/')[-1].strip('_external.txt').split('_')
    #         chip = "l"+chip
    #         with open(s, 'r') as f:
    #             lines = [l.rstrip("\n") for l in f]
    #             registers = []
    #             for i in lines:
    #                 m = re.search('[\t]*(0x[0-9A-F]*)', i)
    #                 registers.append(int(m.group(1), 16),)
    #         self.Config[chip][float(freq)] = registers
        
    
    @property 
    def freq(self): # MHz
        return self.radio.dac_tiles[0].blocks[0].MixerSettings['Freq']
    
    
    @freq.setter
    def freq(self,freq): # MHz
        self.radio.dac_tiles[0].blocks[0].MixerSettings['Freq'] = freq
        
        
        
    def set_i_ampl(self,i_ampl = 0x0000FFFF):
        #set i_ampl (16 bits)
        self.reg[3] = i_ampl
    
    def set_q_ampl(self,q_ampl = 0x0000FFFF):
        #set q_ampl (16 bits)
        self.reg[4] = q_ampl 
        
    def set_phase_difference(self,ph_diff = 0):  # default for 90 degrees
        #set the phase difference to 90 deg
        self.reg[2] = 0x00100000  + int(ph_diff/90*0x00100000)
        

    def set_PDH_phase(self,phase):
        #in machine units (20 bits)
        self.reg[5]= phase

    def set_mod_frequency(self,frequency, sampling_rate = 307_200_000): # Use 430_080_000 for Datapath mode 0    
        frequency_code = int(frequency * 0xFFFF_FFFF / sampling_rate / 8) # divided by 8 because the PDH generator runs 8 times faster
        self.reg[0] = frequency_code

    def change_mod_depth(self,depth=82):
        # 8 bits
        self.reg[1] = depth
        
    def add_offsets(self,I,Q):
        i = np.array(I,dtype=np.int16).tobytes()
        q = np.array(Q,dtype=np.int16).tobytes()
        out = i + q
        self.reg[7] = np.frombuffer(out,dtype=np.uint32)

esbl = ESBL(ext_clock=True)

#esbl.freq = 1000

esbl.change_mod_depth(0)
esbl.set_mod_frequency(5_000_000,sampling_rate = 614_400_000)
esbl.radio.dac_tiles[0].blocks[0].UpdateEvent(1)


def calc_freq(NCOFreq,sampling_rate = 9_830_400_000): # 6_881_280_000 for Datapath mode 0
    XRFDC_NCO_FREQ_MULTIPLIER = np.array(0x1 << 48,dtype = np.uint64)
    freq_list =  np.array( (NCOFreq * XRFDC_NCO_FREQ_MULTIPLIER) // (sampling_rate/2),dtype = np.uint64)
    freq_list[-1] = freq_list[-1] | 1<<63
    return freq_list

In [25]:
# esbl.freq = 2000
esbl.freq = 1500


esbl.change_mod_depth(0)
esbl.change_mod_depth(82)
esbl.set_mod_frequency(3_125_000)
# esbl.set_mod_frequency(3_125_000,sampling_rate = 614_400_000)


esbl.radio.dac_tiles[0].blocks[0].UpdateEvent(1)

In [21]:
try:
    while True:
        esbl.freq = 100
        esbl.radio.dac_tiles[0].blocks[0].UpdateEvent(1)
        esbl.freq = 1000
        esbl.radio.dac_tiles[0].blocks[0].UpdateEvent(1)
except KeyboardInterrupt:
    pass



In [28]:
esbl.init_rf_clks(lmk_freq=245.76, lmx_freq=491.52)
# esbl.init_rf_clks(lmk_freq=245.76, lmx_freq=509.725)

In [82]:
# esbl.set_phase_difference(-3)
# esbl.set_phase_difference(0)
esbl.set_phase_difference(0)
# esbl.radio.dac_tiles[0].blocks[0].UpdateEvent(1)

In [22]:
esbl.set_i_ampl(int(1*0x0000FFFF))
esbl.set_q_ampl(int(1*0x0000FFFF))
esbl.radio.dac_tiles[0].blocks[0].UpdateEvent(1)

In [ ]:
esbl.set_mod_frequency_direct(100000000)
esbl.radio.dac_tiles[0].blocks[0].UpdateEvent(1)

In [24]:
esbl.sync()

In [ ]:
# esbl.freq = 450
# a = esbl.xrfdc_alias[0:65534] + 0
# esbl.freq = 10
# b = esbl.xrfdc_alias[0:65534] + 0
# esbl.freq = 450
# c = esbl.xrfdc_alias[0:65534] + 0
# esbl.freq = 10
# d = esbl.xrfdc_alias[0:65534] + 0


# for n,i in enumerate(a):
#     if (i != b[n]):
#         print(f'{n} ({n*4:x})  {i:} --> {b[n]:x} --> {c[n]:x} --> {d[n]:x}')

In [ ]:
esbl.radio.dac_tiles[0].blocks[0].MixerSettings['EventSource'] = 2

In [75]:
send = esbl.axi_dma_0.sendchannel

In [76]:
from pynq import allocate
import numpy as np

try:
    if send_buffer.size>0:
        raise Error('The buffer is allocated')
except NameError:
    pass
except:
    send_buffer.freebuffer()

data_size = 12_000_000
data_size = 6
# half_size = data_size//2

send_buffer = allocate(shape=(data_size,), dtype=np.uint64)


In [77]:
# send_buffer[:] = np.concatenate([calc_freq(np.linspace(1e6,2e6,data_size//2)),calc_freq(np.linspace(2e6,3e6,data_size//2))])
send_buffer[:] =calc_freq(np.linspace(1e7,1e8,data_size))

In [78]:
print(send_buffer[0:5])
print(send_buffer.shape)
print(send_buffer[-5:])

[ 572662306133 1603454457173 2634246608213 3665038759253 4695830910293]
(6,)
[      1603454457173       2634246608213       3665038759253
       4695830910293 9223377763477837141]


In [79]:
send.transfer(send_buffer)

In [130]:
esbl.freq = 10

esbl.radio.dac_tiles[0].blocks[0].UpdateEvent(1)

In [ ]:
receive.transfer(receive_buffer)

In [ ]:
esbl.radio.dac_tiles[0].blocks[0].SetDACVOP(20000)

In [13]:
esbl.reg[6]=np.array(2**32-1,dtype=np.uint32)
print(esbl.reg[6])
# esbl.reg[6]=np.array(0,dtype=np.uint32)

print(esbl.reg[:32])

4294967295
[   8738133          0    1048576      65535      65535          0
 4294967295          0          0          0          0          0
          0          0          0          0          0          0
          0          0          0          0          0          0
          0          0          0          0          0          0
          0          0]


In [17]:
esbl.ip_dict['ESBL_control_reg']['addr_range']

4096

In [96]:
esbl.reg[6]=1

print(esbl.reg[:32])

[   8738133          0    1048576      65535      65535          0
          1          0          0          0          0          0
          0          0          0          0          0          0
          0          0          0          0          0          0
          1 4294967295 4294967295          1          0          0
          0          0]


In [85]:
esbl.trigger_DMA()

In [5]:
t = Overlay('main.bit')


In [40]:
def calc_freq(NCOFreq,sampling_rate = 9_830_400_000): # 6_881_280_000 for Datapath mode 0
    XRFDC_NCO_FREQ_MULTIPLIER = np.array(0x1 << 48,dtype = np.uint64)
    freq_list =  np.array( (NCOFreq * XRFDC_NCO_FREQ_MULTIPLIER) // (sampling_rate/2),dtype = np.uint64)
    freq_list[-1] = freq_list[-1] | 1<<63
    return freq_list


calc_freq(np.array([1e9,1e9]))

array([     57266230613333, 9223429303085389141], dtype=uint64)

In [7]:
esbl.radio.dac_tiles[0].blocks[0].Co

{'EnablePhase': 0,
 'EnableGain': 0,
 'GainCorrectionFactor': 0.0,
 'PhaseCorrectionFactor': 0.0,
 'OffsetCorrectionFactor': 0,
 'EventSource': 0}

In [78]:
esbl.change_mod_depth(0)
esbl.add_offsets(0x00003FFF,0)#0x00003FFF)
esbl.set_i_ampl(i_ampl=0)# = 0x00007FFF)
esbl.set_q_ampl(q_ampl=0)# = 0x00007FFF)


In [50]:
0x7FFF

32767